# 1. Compute Core Data Products

Refactored notebook version of the core analysis scripts: international citation ratio, citation preferences, publication counts, and reference-list quantities.


In [ ]:
from collections import defaultdict
from pathlib import Path
import gc, pickle, time
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
PICKLE_DIR = PROJECT_ROOT / 'data' / 'Openalex_2025' / 'pickles'
OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'results'
START_YEAR, END_YEAR = 2000, 2025
COUNTRIES = ['US','CN','GB','DE','JP','IT','FR','CA','IN','KR','ES','AU','BR','NL','TR']
COUNTRIES_OT = COUNTRIES + ['OT']
COUNTRIES_TOTAL = COUNTRIES + ['CN_Excluded','Total']
DOMAINS = ['Health Sciences','Life Sciences','Social Sciences','Physical Sciences']

def load_pickle(name):
    with open(PICKLE_DIR / name, 'rb') as fh:
        return pickle.load(fh)

def ensure_dir(path):
    Path(path).mkdir(parents=True, exist_ok=True)

def load_csr_arrays(): # load the citation matrix in CSR format and return the indptr and indices arrays
    mat = load_pickle('citation_matrix_csr.pickle')
    indptr, indices = mat.indptr, mat.indices
    del mat; gc.collect()
    return indptr, indices

def load_wos_valid_set(): # load the set of valid paper IDs that are published in journals indexed by Web of Science (WOS), based on the ISSN matching method
    wos_issn = set(pd.read_csv(PICKLE_DIR / 'wos2025_Journal_ISSN_all.csv')['ISSN'])
    paper_issn = load_pickle('Paper_ISSN_list.pickle')
    valid = {pid for pid, issns in paper_issn.items() if any(x in wos_issn for x in issns)}
    valid.discard('')
    del wos_issn, paper_issn; gc.collect()
    return valid

def refs(indptr, indices, pid): # return the set of reference IDs for a given paper ID
    return set(indices[indptr[pid]:indptr[pid+1]])

def norm_country(country, country_to_idx): # return the normalized country name if it is in the country_to_idx mapping, otherwise return 'OT' for other
    return country if country in country_to_idx else 'OT'

def write_year_country(table, out_path, years, countries):
    ensure_dir(Path(out_path).parent)
    with open(out_path, 'w', encoding='utf-8') as fh:
        fh.write('year\t' + ''.join(f'{c}\t' for c in countries) + '\n')
        for year in years:
            fh.write(f'{year}\t' + ''.join(f'{table[year][c]}\t' for c in countries) + '\n')

def write_array(data, out_path, years, cols):
    ensure_dir(Path(out_path).parent)
    with open(out_path, 'w', encoding='utf-8') as fh:
        fh.write('year\t' + ''.join(f'{c}\t' for c in cols) + '\n')
        for i, year in enumerate(years):
            fh.write(f'{year}\t' + ''.join(f'{data[i,j]}\t' for j in range(len(cols))) + '\n')

def write_matrix_by_year(data, out_dir, prefix, years, countries):
    ensure_dir(out_dir)
    for year in years:
        with open(Path(out_dir)/f'{prefix}_{year}.txt', 'w', encoding='utf-8') as fh:
            fh.write('\\i(\\g(\\ab(d))\\-(i,j))\t' + ''.join(f'{c}\t' for c in countries) + '\n')
            for citing_ctry in countries:
                fh.write(f'{citing_ctry}\t')
                for cited_ctry in countries:
                    value = 0 if citing_ctry == cited_ctry else data[year][citing_ctry][cited_ctry]
                    fh.write(f'{value}\t')
                fh.write('\n')


## International citation ratio $f$

Functions for calculating $f$ under different circumstances.

In [ ]:
def cited_paper_tags(pid, first_country, corr_country, single_country, oa, domain, corr_mode=False):
    tags = ['Total']
    if pid in corr_country and pid in first_country:
        tags.append('Sole-1st-corr' if corr_country[pid] == first_country[pid] else 'Multi-1st-corr')
    tags.append('Sole-country' if pid in single_country else 'Multi-country')
    tags.append('OA' if oa.get(pid, False) else 'Non-OA')
    if pid in domain:
        tags.append(domain[pid])
    return tags

def compute_f(corr_mode=False):
    t0 = time.time()
    indptr, indices = load_csr_arrays() # load the citation matrix in CSR format and get the indptr and indices arrays
    valid = load_wos_valid_set() # load the set of valid paper IDs that are published in journals indexed by Web of Science (WOS), based on the ISSN matching method
    first_country = load_pickle('Paper_country_1st_aff.pickle') # load the mapping of paper ID to the country of the first affiliation
    corr_country = load_pickle('Paper_country_corr_aff.pickle') # load the mapping of paper ID to the country of the corresponding affiliation
    all_country = load_pickle('Paper_country_all_aff.pickle') # load the mapping of paper ID to the list of countries of all affiliations
    single_country = {pid for pid, countries in all_country.items() if len(set(countries)) == 1} # get the set of paper IDs that have only one unique country in all affiliations
    paper_year = load_pickle('Paper_year.pickle') # load the mapping of paper ID to publication year
    oa = load_pickle('Paper_OA_tag.pickle') # load the mapping of paper ID to open access (OA) status
    domain = load_pickle('Paper_domain_1st.pickle') # load the mapping of paper ID to the primary scientific domain
    categories = ['Total','Sole-1st-corr','Multi-1st-corr','Sole-country','Multi-country','OA','Non-OA'] + DOMAINS # define the categories for cited papers based on the corresponding author country, the number of countries involved, the OA status, and the scientific domain
    
    if corr_mode:
        country_map = corr_country
        out_dir = OUTPUT_ROOT / 'p1_inter_citation_rate_corr'
    else:
        country_map = first_country
        out_dir = OUTPUT_ROOT / 'p1_inter_citation_rate'
    years = range(START_YEAR, END_YEAR)
    counts = {T:{c:{y:{cat:[0,0] for cat in categories} for y in years} for c in COUNTRIES} for T in [10,5,3]} # initialize the counts dictionary to store the citation counts for each cited country, year, category, and time interval (10, 5, 3 years), where the value is a list of [inter_count, total_count] for calculating the ICR later
    for n, (pid, y) in enumerate(paper_year.items(), 1):
        if pid not in valid or not (START_YEAR <= y < END_YEAR) or pid not in country_map: continue # filter papers that are not valid (WOS indexed), not in the target year range, or do not have country information
        if n % 100000 == 0: print('processed', n)
        citing_ctry = country_map[pid] # get the country of the citing paper without 
        for rid in refs(indptr, indices, pid): # iterate over the reference IDs of the citing paper
            if rid not in valid: continue
            ry, cited_ctry = paper_year.get(rid), country_map.get(rid)
            if ry is None or cited_ctry not in COUNTRIES or not (y-10 <= ry < y): continue
            slots = [1] + ([0] if cited_ctry != citing_ctry else []) # define the slots for counting based on whether the cited country is the same as the citing country, where slot 1 is for all citations and slot 0 is for international citations only
            for tag in cited_paper_tags(rid, first_country, corr_country, single_country, oa, domain, corr_mode):
                if tag not in categories: continue
                for slot in slots:
                    counts[10][cited_ctry][y][tag][slot] += 1
                    if ry >= y-5: counts[5][cited_ctry][y][tag][slot] += 1
                    if ry >= y-3: counts[3][cited_ctry][y][tag][slot] += 1
    ensure_dir(out_dir)
    for T, table in counts.items():
        for c in COUNTRIES:
            for y in years:
                for tag in categories:
                    inter, total = table[c][y][tag]
                    table[c][y][tag] = inter / (total if total else 1)
        for tag in categories:
            for normalized in [False, True]:
                out = out_dir / f"ICR{'_norm' if normalized else ''}_of_Country_in_{tag}_interval{T}.txt"
                with open(out, 'w', encoding='utf-8') as fh:
                    fh.write('year\t' + ''.join(f'{c}\t' for c in COUNTRIES) + '\n')
                    for y in years:
                        fh.write(f'{y}\t')
                        for c in COUNTRIES:
                            base, value = table[c][START_YEAR][tag], table[c][y][tag]
                            fh.write('\t' if base == 0 else f'{value/base if normalized else value}\t')
                        fh.write('\n')
    print('time cost', time.time()-t0, 's')


Call the function to calculate the corresponding $f$.

In [ ]:
compute_f(corr_mode=False)

In [ ]:
compute_f(corr_mode=True)

## Citation preferences
Functions for calculating $\Delta_i$ and $\Delta_{j,i}$.

In [ ]:
def compute_domestic_preference():
    t0=time.time(); indptr, indices = load_csr_arrays(); valid = load_wos_valid_set()
    country = load_pickle('Paper_country_1st_aff.pickle'); year = load_pickle('Paper_year.pickle')
    years=range(START_YEAR,END_YEAR); out=OUTPUT_ROOT/'p3_domestic_referencing_rate_delta'; ensure_dir(out)
    stock={c:{y:{'Total':0} for y in years} for c in COUNTRIES+['Global']}
    raw={c:{y:{'Total':[0,0]} for y in years} for c in COUNTRIES}
    delta={c:{y:{'Total':0.0} for y in years} for c in COUNTRIES}
    for n,(pid,y) in enumerate(year.items(),1):
        if pid not in valid: continue
        c=country.get(pid)
        if y in range(START_YEAR-10, END_YEAR-1) and c is not None:
            for fy in range(max(y+1,START_YEAR), min(y+11,END_YEAR)):
                stock['Global'][fy]['Total'] += 1
                if c in COUNTRIES: stock[c][fy]['Total'] += 1
        if not (START_YEAR <= y < END_YEAR) or c not in COUNTRIES: continue
        d=t=0
        for rid in refs(indptr,indices,pid):
            if rid not in valid: continue
            ry, rc = year.get(rid), country.get(rid)
            if ry is not None and rc is not None and y-10 <= ry < y:
                t += 1; d += int(rc == c)
        if t:
            raw[c][y]['Total'][0] += d/t; raw[c][y]['Total'][1] += 1
    for c in COUNTRIES:
        for y in years:
            stock[c][y]['Total'] = stock[c][y]['Total'] / (stock['Global'][y]['Total'] or 1)
            s,n = raw[c][y]['Total']; raw[c][y]['Total'] = s / (n if n else 1)
            delta[c][y]['Total'] = raw[c][y]['Total'] - stock[c][y]['Total']
    for name,data in [('paper_global_share_10year',stock),('domestic_referencing_rate_mean',raw),('domestic_referencing_rate_relatively_mean',delta)]:
        with open(out/f'{name}_of_Country_in_Total.txt','w',encoding='utf-8') as fh:
            fh.write('year\t'+''.join(f'{c}\t' for c in COUNTRIES)+'\n')
            for y in years: fh.write(f'{y}\t'+''.join(f'{data[c][y]["Total"]}\t' for c in COUNTRIES)+'\n')
        for c in COUNTRIES:
            with open(out/f'{name}_of_Field_in_{c}.txt','w',encoding='utf-8') as fh:
                fh.write('year\tTotal\t\n')
                for y in years: fh.write(f'{y}\t{data[c][y]["Total"]}\t\n')
    print('time cost', time.time()-t0, 's')

def compute_international_preference(Q1_only=False):
    t0=time.time(); indptr,indices=load_csr_arrays(); valid=load_wos_valid_set()
    country=load_pickle('Paper_country_1st_aff.pickle'); year=load_pickle('Paper_year.pickle')
    rank=load_pickle('Paper_ISSN_SJR.pickle') if Q1_only else {}
    out=OUTPUT_ROOT/('p4_inter_referencing_rate_Q1' if Q1_only else 'p4_inter_referencing_rate'); ensure_dir(out)
    years=range(START_YEAR,END_YEAR)
    stock={y:defaultdict(int) for y in years}
    raw={y:{j:{i:[0,0] for i in COUNTRIES} for j in COUNTRIES} for y in years}
    exp={y:{j:defaultdict(float) for j in COUNTRIES} for y in years}; delta={y:{j:defaultdict(float) for j in COUNTRIES} for y in years}
    for n,(pid,y) in enumerate(year.items(),1):
        if pid not in valid: continue
        c=country.get(pid)
        stock_ok = y in range(START_YEAR-10,END_YEAR-1) and c is not None and ((not Q1_only) or rank.get(pid)=='Q1')
        if stock_ok:
            cc=c if c in COUNTRIES else 'OT'
            for fy in range(max(y+1,START_YEAR), min(y+11,END_YEAR)):
                stock[fy][cc]+=1; stock[fy]['Global']+=1
        if not (START_YEAR <= y < END_YEAR) or c not in COUNTRIES: continue
        inter=defaultdict(int); total=0
        for rid in refs(indptr,indices,pid):
            if rid not in valid or (Q1_only and rank.get(rid)!='Q1'): continue
            ry, rc = year.get(rid), country.get(rid)
            if ry is None or rc is None or not (y-10 <= ry < y): continue
            rc = rc if rc in COUNTRIES else 'OT'
            if rc != c: inter[rc]+=1; total+=1
        if total:
            for cited_ctry in COUNTRIES:
                if cited_ctry == c: continue
                raw[y][c][cited_ctry][0] += inter[cited_ctry]/total; raw[y][c][cited_ctry][1] += 1
    for y in years:
        for citing_ctry in COUNTRIES:
            den = stock[y]['Global'] - stock[y][citing_ctry] or 1
            for cited_ctry in COUNTRIES:
                s,n = raw[y][citing_ctry][cited_ctry]; raw[y][citing_ctry][cited_ctry]=s/(n if n else 1)
                exp[y][citing_ctry][cited_ctry]=stock[y][cited_ctry]/den
                delta[y][citing_ctry][cited_ctry]=raw[y][citing_ctry][cited_ctry]-exp[y][citing_ctry][cited_ctry]
    write_year_country(stock, out/'paper_global_share_10year_of_Country_in_Total.txt', years, COUNTRIES)
    write_matrix_by_year(exp,out,'inter_expected_referencing_rate_matrix_interval10',years,COUNTRIES)
    write_matrix_by_year(raw,out,'inter_raw_referencing_rate_matrix_interval10',years,COUNTRIES)
    write_matrix_by_year(delta,out,'inter_referencing_rate_delta_matrix_interval10',years,COUNTRIES)
    print('time cost', time.time()-t0, 's')


Call the function to calculate $\Delta_i$ or $\Delta_{j,i}$.

In [ ]:
compute_domestic_preference()

In [ ]:
compute_international_preference(Q1_only=False)

In [ ]:
compute_international_preference(Q1_only=True)

## Publication and reference growth quantities
Functions for calculating publication and reference growth quantities.

In [ ]:
def compute_publication_counts_and_growth():
    t0=time.time(); valid=load_wos_valid_set(); country=load_pickle('Paper_country_1st_aff.pickle'); year=load_pickle('Paper_year.pickle')
    years=range(START_YEAR,END_YEAR); counts={y:defaultdict(int) for y in years}
    for pid,y in year.items():
        if pid not in valid or y not in years or pid not in country: continue
        c=country[pid]; counts[y][c]+=1; counts[y]['Total']+=1
        if c!='CN': counts[y]['CN_Excluded']+=1
    shares={y:defaultdict(float) for y in years}
    for y in years:
        total=counts[y]['Total'] or 1
        for c in COUNTRIES_TOTAL: shares[y][c]=counts[y][c]/total
    growth={y:defaultdict(float) for y in range(START_YEAR+1,END_YEAR)}
    for y in range(START_YEAR+1,END_YEAR):
        for c in COUNTRIES_TOTAL: growth[y][c]=counts[y][c]/counts[y-1][c]-1
    write_year_country(counts, OUTPUT_ROOT/'p5_reduce_paper_count/Paper_Count_of_Country.txt', years, COUNTRIES_TOTAL)
    write_year_country(shares, OUTPUT_ROOT/'p5_reduce_paper_count/Paper_Share_of_Country.txt', years, COUNTRIES_TOTAL)
    write_year_country(growth, OUTPUT_ROOT/'p5_reduce_paper_rate/Paper_Growth_Rate_of_Country.txt', range(START_YEAR+1,END_YEAR), COUNTRIES_TOTAL)
    print('time cost',time.time()-t0,'s')

def compute_reference_list_quantities():
    t0=time.time(); indptr,indices=load_csr_arrays(); valid=load_wos_valid_set()
    country=load_pickle('Paper_country_1st_aff.pickle'); year=load_pickle('Paper_year.pickle')
    years=range(START_YEAR,END_YEAR); out=OUTPUT_ROOT/'p6_reference_list_length'; ensure_dir(out)
    R_L={y:defaultdict(int) for y in years}; R_open={y:defaultdict(int) for y in years}; R10={y:defaultdict(int) for y in years}; C10={y:defaultdict(int) for y in years}; paper_count={y:defaultdict(int) for y in years}
    for n,(pid,y) in enumerate(year.items(),1):
        if pid not in valid or not (START_YEAR<=y<END_YEAR) or pid not in country: continue
        if n%100000==0: print('processed',n)
        c=country[pid]; rset=refs(indptr,indices,pid); L=len(rset); ropen=0; r10=0
        for rid in rset:
            if rid not in valid: continue
            ropen += 1; ry,rc=year.get(rid),country.get(rid)
            if ry is not None and rc is not None and y-10 <= ry < y:
                r10 += 1; C10[y][rc]+=1; C10[y]['Total']+=1
        keys=['Total'] + (['CN_Excluded'] if c!='CN' else []) + ([c] if c in COUNTRIES_TOTAL else [])
        for key in keys:
            R_L[y][key]+=L; R_open[y][key]+=ropen; R10[y][key]+=r10; paper_count[y][key]+=1
    records=[]
    for y in years:
        for c in COUNTRIES_TOTAL:
            n=paper_count[y][c]
            records.append({'Year':y,'Country':c,'reference_count_R_L':R_L[y][c],'reference_count_R_open':R_open[y][c],'reference_count_R10':R10[y][c],'paper_count':n,'ave_R_L':R_L[y][c]/n,'ave_R_open':R_open[y][c]/n,'ave_R10':R10[y][c]/n,'citation_count':C10[y][c]})
    df=pd.DataFrame(records)
    def write_count(tag):
        with open(out/f'Country_{tag}.txt','w',encoding='utf-8') as fh:
            fh.write('year\t'+''.join(f'{c}\t' for c in COUNTRIES_TOTAL)+'\n')
            for y in years:
                fh.write(f'{y}\t')
                for c in COUNTRIES_TOTAL: fh.write(f"{df[(df.Year==y)&(df.Country==c)][tag].tolist()[0]}\t")
                fh.write('\n')
    def write_growth(tag,suffix):
        with open(out/f'Country_reference_{suffix}_growth_rate.txt','w',encoding='utf-8') as fh:
            fh.write('year\t'+''.join(f'{c}\t' for c in COUNTRIES_TOTAL)+'\n')
            for y in range(START_YEAR+1,END_YEAR):
                fh.write(f'{y}\t')
                for c in COUNTRIES_TOTAL:
                    prev=df[(df.Year==y-1)&(df.Country==c)][tag].tolist()[0]; cur=df[(df.Year==y)&(df.Country==c)][tag].tolist()[0]
                    fh.write(f'{cur/prev-1}\t')
                fh.write('\n')
    for suffix in ['R_L','R_open','R10']:
        write_count(f'reference_count_{suffix}'); write_count(f'ave_{suffix}'); write_growth(f'reference_count_{suffix}',suffix)
    write_count('citation_count')
    print('time cost',time.time()-t0,'s')


Call the functions to compute.

In [ ]:
compute_publication_counts_and_growth()

In [ ]:
compute_reference_list_quantities()